In [ ]:
import random
from datasets import load_dataset, Dataset
import time

# ==============================================================================
# 🚀 [데이터셋 정보]
# 제목: 한국어 감성 분석 데이터셋 (Korean Sentiment)
# 의미: 다양한 문장에 대해 긍정적, 부정적 감성을 분류하여 학습할 수 있는 데이터셋입니다.
# 설명: AI가 글의 감정을 파악하는 '감성 분석(Sentiment Analysis)' 능력을 키우는 실습에 최적화되어 있습니다.
# ==============================================================================

# 🌟 코딩 튜터님의 오늘의 미션! 🌟
# 안녕하세요, 코딩 탐험가님! 오늘은 AI가 글의 마음을 읽어내는 '감성 분석'의 세계로 떠나 볼 거예요.
# 데이터셋의 구조를 파악하고, 재미있는 코드를 작성하며 실력을 쑥쑥 키워 봅시다!

DATASET_NAME = "sepidmnorozy/Korean_sentiment"
SAMPLE_COUNT = 50 # 너무 많은 데이터를 로드하면 느려질 수 있으니, 일단 50개만 볼게요!

# ------------------------------------------------------------------------------
# 💾 1. 데이터셋 로드 (스트리밍 최적화!)
# 스트리밍 로드는 대용량 데이터를 메모리 부담 없이 빠르게 읽어오는 고급 기술이랍니다.
# ------------------------------------------------------------------------------

print("=================================================================")
print("✅ 1. 데이터셋 로드 시도 (스트리밍 모드)")
print("=================================================================")

dataset = None
try:
    # 테스트 데이터셋에서 스트리밍을 시도합니다.
    dataset = load_dataset(DATASET_NAME, split='test', streaming=True)
    print("✨ 성공! 스트리밍(Streaming) 모드로 데이터셋을 로드했습니다. 메모리 걱정 NO!")

except Exception as e:
    # 만약 스트리밍이 실패하면, 일반 Dataset 객체로 fallback합니다.
    print(f"⚠️ 스트리밍 로드 실패 ({type(e).__name__}) - 일반 모드로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='test', streaming=False)
        print("✨ 성공! 일반 Dataset 모드로 테스트 스플릿을 로드했습니다.")
    except Exception as e_fallback:
        print(f"🚨 치명적인 오류 발생: 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()


# ------------------------------------------------------------------------------
# 📚 2. 샘플 데이터 준비 (전체 데이터를 로드하지 마세요!)
# Constraint: 전체 데이터셋을 사용하지 않고, 상위 K개만 가져와요!
# ------------------------------------------------------------------------------

print("\n=================================================================")
print(f"🤖 2. 실습용 데이터 {SAMPLE_COUNT}개 준비 중...")
print("=================================================================")

# 데이터를 list(dataset.take(K)) 패턴으로 안전하게 샘플링합니다.
sampled_dataset = dataset.take(SAMPLE_COUNT)
sample_data_list = list(sampled_dataset)

if not sample_data_list:
    print("❌ 샘플 데이터를 가져올 수 없습니다. 데이터셋의 구조를 확인해 주세요.")
    exit()

print(f"✅ 준비 완료! 총 {len(sample_data_list)}개의 샘플을 가져와 실습에 사용합니다.")


# ------------------------------------------------------------------------------
# ✨ 3. 실습 1: 데이터 구조 살펴보기 (질문과 답안 매칭하기)
# ------------------------------------------------------------------------------

print("\n\n=================================================================")
print("🧠 3. 실습 1: 감성 분석 데이터 구조 탐험")
print("=================================================================")
print("💡 데이터는 ['text']: 문장, ['label']: 감성 점수로 구성되어 있어요.")

# 첫 2개의 샘플을 뽑아서 눈으로 확인해 봅시다.
print("\n--- 샘플 2개 분석 ---")
for i in range(2):
    sample = sample_data_list[i]
    text = sample['text']
    label = sample['label']
    print(f"  [샘플 {i+1}] ➡️ 문장: '{text[:40]}...'")
    print(f"         ⭐ 예측된 감성(Label): {label} (0: 부정/1: 중립/2: 긍정 추정)")

# 📊 정량적 분석: 라벨 분포 확인하기
print("\n--- ✨ 라벨 분포 분석 (개요 파악) ---")
label_counts = {}
for sample in sample_data_list:
    label = sample['label']
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"🔍 분석 결과: 총 {len(sample_data_list)}개의 샘플 중:")
print(f"   - Label 0 (부정): {label_counts.get(0, 0)} 건")
print(f"   - Label 1 (중립): {label_counts.get(1, 0)} 건")
print(f"   - Label 2 (긍정): {label_counts.get(2, 0)} 건")


# ------------------------------------------------------------------------------
# 💻 4. 실습 2: 간단한 감성 분류 시뮬레이터 만들기 (창의적 실습)
# ------------------------------------------------------------------------------

print("\n\n=================================================================")
print("💡 4. 실습 2: 나만의 감성 분류기 시뮬레이션!")
print("=================================================================")
print("🤖 문장만 보고 감성을 예측하는 함수를 만들어 볼 거예요!")

def simple_sentiment_predictor(text):
    """
    매우 단순하게 키워드를 기반으로 감성을 예측하는 토이 함수입니다.
    (AI가 진짜 쓰는 방식은 아니지만, 원리를 이해하는 게 중요해요!)
    """
    text_lower = text.lower()
    score = 0

    # 긍정 키워드 체크
    if "좋다" in text_lower or "최고" in text_lower or "행복" in text_lower:
        score += 2
    # 부정 키워드 체크
    if "싫다" in text_lower or "안 좋아" in text_lower or "최악" in text_lower:
        score -= 2
    # 중립 키워드 체크
    if "알다" in text_lower or "하다" in text_lower:
        score += 0

    if score > 0:
        return "✅ 긍정적 (Positive)"
    elif score < 0:
        return "❌ 부정적 (Negative)"
    else:
        return "🟡 중립적 (Neutral)"

# 테스트 문장들
test_sentences = [
    "오늘 날씨가 정말 최고로 좋아서 기분이 아주 행복해요.", # 긍정
    "이 영화는 기대와 달리 너무 안 좋아 가지고 실망했어요.", # 부정
    "내일 회의는 3시에 예정되어 있어.", # 중립
    "이 데이터셋의 활용법을 배우는 것은 정말 좋다." # 복합 (긍정)
]

print("\n--- 🌟 테스트 문장으로 감성 예측 ---")
for sentence in test_sentences:
    prediction = simple_sentiment_predictor(sentence)
    print(f"   ➡️ '{sentence}'\n   --> 예측 감성: {prediction}")

# ------------------------------------------------------------------------------
# 📝 5. 실습 3: LLM (대형 언어 모델) 프롬프트 설계하기
# ------------------------------------------------------------------------------

print("\n\n=================================================================")
print("✍️ 5. 실습 3: AI에게 '임무'를 부여하는 프롬프트 설계")
print("=================================================================")
print("✨ 데이터 분석의 마지막 관문은, AI가 우리가 원하는 형태로 답하게 만드는 것입니다!")

def generate_prompt(example_text):
    """
    감성 분석 태스크를 수행할 수 있도록 LLM에게 요청하는 프롬프트를 생성합니다.
    """
    prompt = f"""
    [사용자에게 요청] 다음 한국어 문장의 감성을 분석하고, 반드시 '점수'와 '감성 분류'를 포함하여 JSON 형식으로 응답해 주세요.
    감성 점수는 -2 (매우 부정)부터 +2 (매우 긍정) 사이의 정수여야 합니다.
    
    문장: "{example_text}"
    
    응답 형식 (JSON):
    {{
      "text": "{example_text}",
      "sentiment_score": [최종 점수],
      "sentiment_class": "[Positive | Negative | Neutral]"
    }}
    """
    return prompt

# 샘플 데이터를 활용하여 프롬프트 생성 및 출력
sample_text = sample_data_list[0]['text']
generated_prompt = generate_prompt(sample_text)

print("\n--- 💌 LLM에 전달할 마법의 프롬프트 ---")
print("----------------------------------------------------------------")
print(generated_prompt)
print("----------------------------------------------------------------")
print("\n✨ 튜터 코멘트: 이렇게 프롬프트를 설계하면, AI는 정확한 형식(JSON)으로 답하려고 노력할 거예요!")


print("\n\n=================================================================")
print("🎉 축하합니다! 데이터셋 분석의 기초부터 AI 프롬프트 설계까지 모두 마쳤어요!")
print("👏 원리 이해가 가장 중요합니다. 다음 실습도 화이팅입니다!")
print("=================================================================")